# 10.07 - Optimization for CV

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** a CV optimization ablation comparing three learning-rate and weight-decay settings on the same small CNN.

Today is about controlling training instead of only making a model bigger: optimizer choice, learning rate, scheduler behavior, weight decay, and reading train/validation curves for overfitting signals.


## Core Ideas

Optimization choices decide how a CNN moves through the loss landscape.

- **Learning rate:** the step size. Too small can underfit slowly; too large can bounce or diverge.
- **Optimizer:** SGD with momentum is simple and strong; Adam adapts per-parameter step sizes and often learns quickly.
- **Weight decay:** penalizes large weights and can reduce overfitting.
- **Scheduler:** changes learning rate over epochs, often lowering it after the model reaches a plateau.
- **Overfitting curves:** training loss keeps improving while validation loss stops improving or gets worse.

For a fair ablation, keep the dataset, architecture, batch size, seed, and number of epochs fixed. Change only the optimizer settings you are testing.


In [ ]:
import random
import numpy as np

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False
    print("PyTorch is not installed. Complete this notebook in an environment with torch.")

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)

set_seed(SEED)
if TORCH_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
else:
    device = None


## Prepared Image Data

Run this cell before the exercises. The synthetic optimization dataset is provided so the ablation work can focus on DataLoaders, the CNN, optimizer settings, and curve analysis.


In [ ]:
def make_optimization_dataset(n_per_class=60, image_size=16, noise=0.14, seed=42):
    if not TORCH_AVAILABLE:
        raise ImportError("PyTorch is required for this exercise.")

    generator = torch.Generator().manual_seed(seed)
    images = []
    labels = []

    for class_id in range(3):
        for _ in range(n_per_class):
            img = torch.zeros(1, image_size, image_size, dtype=torch.float32)
            center = image_size // 2
            if class_id == 0:
                img[:, :, center - 2:center] = 1.0
                img[:, :, center + 3:center + 4] = 0.65
            elif class_id == 1:
                img[:, center - 2:center, :] = 1.0
                img[:, center + 3:center + 4, :] = 0.65
            else:
                for i in range(image_size):
                    img[:, i, i] = 1.0
                    if i + 1 < image_size:
                        img[:, i, i + 1] = 0.75

            img = img + noise * torch.randn(img.shape, generator=generator)
            images.append(img.clamp(0.0, 1.0))
            labels.append(class_id)

    X = torch.stack(images)
    y = torch.tensor(labels, dtype=torch.long)
    perm = torch.randperm(len(y), generator=generator)
    return X[perm], y[perm]


if TORCH_AVAILABLE:
    X, y = make_optimization_dataset()
    print("X:", X.shape, X.dtype)
    print("y:", y.shape, y.dtype, sorted(y.unique().tolist()))


## Exercise 10-A: DataLoaders and CNN Setup

Use the prepared tensors `X` and `y`. Build deterministic train/validation loaders and a compact CNN that outputs logits shaped `[batch, num_classes]`.


In [ ]:
# TODO 10-A

def build_loaders(X, y, batch_size=32, train_frac=0.75, seed=42):
    raise NotImplementedError("Split X/y into train and validation DataLoaders.")


class TinyOptimizationCNN(nn.Module if TORCH_AVAILABLE else object):
    def __init__(self, num_classes=3):
        if not TORCH_AVAILABLE:
            return
        super().__init__()
        raise NotImplementedError("Build a compact CNN classifier.")

    def forward(self, x):
        raise NotImplementedError("Return logits shaped [batch, num_classes].")


## Exercise 10-B: Optimizer and Scheduler Builder

Write a config-driven optimizer builder. It should support at least `adam` and `sgd`, learning rate, weight decay, SGD momentum, and optional schedulers.

Recommended scheduler options:

- `none`: keep LR constant
- `step`: multiply LR by `gamma` every `step_size` epochs
- `cosine`: cosine decay over the planned number of epochs


In [ ]:
# TODO 10-B

def build_optimizer_and_scheduler(model, config):
    raise NotImplementedError("Return (optimizer, scheduler) from a config dictionary.")


def get_current_lr(optimizer):
    raise NotImplementedError("Return the learning rate from the first optimizer parameter group.")


## Exercise 10-C: Train One Experiment

Implement the reusable training functions. Each epoch should record:

- `epoch`
- `train_loss`
- `train_acc`
- `val_loss`
- `val_acc`
- `lr`

Call `model.train()` during training and `model.eval()` plus `torch.no_grad()` during validation.


In [ ]:
# TODO 10-C

def train_one_epoch(model, loader, criterion, optimizer, device):
    raise NotImplementedError("Train for one epoch and return average loss and accuracy.")


def evaluate(model, loader, criterion, device):
    raise NotImplementedError("Evaluate and return a dict with loss and accuracy.")


def run_training_experiment(config, train_loader, val_loader, epochs=4, seed=42, device=device):
    raise NotImplementedError("Create a fresh model, train it, and return a result dictionary with history.")


## Exercise 10-D: Run a Three-Setting Ablation

Run three optimizer settings on the same CNN and compare the curves.

Include one deliberately aggressive setting, one Adam setting with weight decay, and one SGD setting with momentum. Store enough information to identify the best validation result.


In [ ]:
# TODO 10-D

ABLATION_CONFIGS = [
    # Example shape:
    # {"name": "adam_lr_1e-2_no_decay", "optimizer": "adam", "lr": 1e-2, "weight_decay": 0.0, "scheduler": "none"},
]


def run_ablation(configs, train_loader, val_loader, epochs=4, seed=42, device=device):
    raise NotImplementedError("Run every config and return a list of experiment result dictionaries.")


def rank_ablation_results(results):
    raise NotImplementedError("Return a compact summary sorted by best validation loss.")


## Exercise 10-E: Diagnose Optimization Curves

Turn the history into a short diagnosis. Look for:

- overfitting: high train accuracy, lower validation accuracy, and validation loss rebound
- underfitting: both train and validation accuracy remain low
- improving: validation loss is moving down by the end
- unstable: validation loss is not clearly improving


In [ ]:
# TODO 10-E

def diagnose_curve(history, gap_threshold=0.15, loss_rebound=0.05):
    raise NotImplementedError("Return a dictionary with label, accuracy gap, best val loss, and final val loss.")


def make_ablation_report(results):
    raise NotImplementedError("Combine ranking and curve diagnosis into a report list.")


## Test Cases

Run this cell after completing the TODO cells above. A correct implementation should print `Day 10 tests passed`.


In [ ]:
def run_day10_tests():
    if not TORCH_AVAILABLE:
        print("Day 10 tests skipped because PyTorch is not installed.")
        return

    required_names = [
        "make_optimization_dataset",
        "build_loaders",
        "TinyOptimizationCNN",
        "build_optimizer_and_scheduler",
        "get_current_lr",
        "train_one_epoch",
        "evaluate",
        "run_training_experiment",
        "run_ablation",
        "rank_ablation_results",
        "diagnose_curve",
        "make_ablation_report",
    ]
    for name in required_names:
        assert name in globals(), f"Missing function or class: {name}"
        assert callable(globals()[name]), f"{name} must be callable"

    X_test, y_test = make_optimization_dataset(n_per_class=6, image_size=16, seed=123)
    assert X_test.shape == (18, 1, 16, 16), f"Unexpected X shape: {X_test.shape}"
    assert y_test.shape == (18,), f"Unexpected y shape: {y_test.shape}"
    assert X_test.dtype == torch.float32
    assert y_test.dtype == torch.long
    assert set(y_test.tolist()) == {0, 1, 2}

    train_loader_test, val_loader_test = build_loaders(X_test, y_test, batch_size=6, train_frac=0.67, seed=123)
    xb, yb = next(iter(train_loader_test))
    assert xb.ndim == 4 and xb.shape[1:] == (1, 16, 16)
    assert yb.dtype == torch.long

    model_test = TinyOptimizationCNN(num_classes=3).to(device)
    with torch.no_grad():
        logits = model_test(xb.to(device))
    assert logits.shape == (xb.shape[0], 3), f"Unexpected logits shape: {logits.shape}"

    adam_config = {
        "name": "test_adam",
        "optimizer": "adam",
        "lr": 0.003,
        "weight_decay": 1e-4,
        "scheduler": "step",
        "step_size": 1,
        "gamma": 0.5,
    }
    optimizer, scheduler = build_optimizer_and_scheduler(model_test, adam_config)
    assert isinstance(optimizer, torch.optim.Optimizer)
    assert abs(get_current_lr(optimizer) - 0.003) < 1e-12
    if scheduler is not None:
        optimizer.step()
        scheduler.step()
        assert get_current_lr(optimizer) < 0.003

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model_test.parameters(), lr=0.003)
    train_loss, train_acc = train_one_epoch(model_test, train_loader_test, criterion, optimizer, device)
    assert isinstance(train_loss, float)
    assert 0.0 <= train_acc <= 1.0

    metrics = evaluate(model_test, val_loader_test, criterion, device)
    assert {"loss", "accuracy"}.issubset(metrics.keys())
    assert isinstance(metrics["loss"], float)
    assert 0.0 <= metrics["accuracy"] <= 1.0

    quick_configs = [
        {"name": "quick_adam", "optimizer": "adam", "lr": 0.003, "weight_decay": 0.0, "scheduler": "none"},
        {"name": "quick_sgd", "optimizer": "sgd", "lr": 0.03, "momentum": 0.9, "weight_decay": 1e-4, "scheduler": "step", "step_size": 1, "gamma": 0.5},
    ]
    results = run_ablation(quick_configs, train_loader_test, val_loader_test, epochs=2, seed=123, device=device)
    assert len(results) == 2
    for result in results:
        assert {"name", "config", "history", "best_val_loss", "best_val_acc", "final_val_loss", "final_val_acc"}.issubset(result.keys())
        assert len(result["history"]) == 2
        assert {"epoch", "train_loss", "train_acc", "val_loss", "val_acc", "lr"}.issubset(result["history"][0].keys())

    ranked = rank_ablation_results(results)
    assert len(ranked) == 2
    assert ranked[0]["best_val_loss"] <= ranked[-1]["best_val_loss"]

    overfit_history = [
        {"train_loss": 0.50, "train_acc": 0.80, "val_loss": 0.40, "val_acc": 0.78},
        {"train_loss": 0.15, "train_acc": 0.99, "val_loss": 0.70, "val_acc": 0.62},
    ]
    underfit_history = [
        {"train_loss": 1.20, "train_acc": 0.35, "val_loss": 1.25, "val_acc": 0.30},
        {"train_loss": 1.10, "train_acc": 0.45, "val_loss": 1.18, "val_acc": 0.38},
    ]
    assert diagnose_curve(overfit_history)["label"] == "overfitting"
    assert diagnose_curve(underfit_history)["label"] == "underfitting"

    report = make_ablation_report(results)
    assert len(report) == 2
    assert {"name", "best_val_loss", "best_val_acc", "final_val_acc", "diagnosis"}.issubset(report[0].keys())

    print("Day 10 tests passed")

run_day10_tests()


## Day 10 Checklist

Before trusting an optimization ablation, verify that all runs use the same model and data split, each config records learning rate and weight decay, validation is measured with `eval()` and `no_grad()`, the best run is selected by validation metrics, and curve diagnosis is based on both train and validation behavior.
